# 跳包模型

这里是跳包模型的完整实现。我们要把那个“玩具模型”扔进垃圾桶，因为它有两个致命问题，导致无法处理数百万词汇量的真实语料：

1. Softmax 计算量太大：每次预测都要计算几万个词的概率，词表一大，训练速度会慢到让你怀疑人生。
2. 高频词干扰：像 "the" 这种词出现几百万次，如果不处理，模型会花费大量时间去学 "the" 和其他词的关系，既浪费时间又拉低精度。

我们要实现的是 带有负采样 (Negative Sampling) 和 下采样 (Sub-sampling) 的 Skip-gram。这是 Google 在 Word2Vec 论文中使用的真正架构。

我们将使用 WikiText-2 数据集（维基百科文章集合），这是一个标准的 NLP 工业级基准数据集。

## 第一步：环境配置与数据下载

我们需要处理真实文件。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import collections
import random
import math
import time

# 检查是否有可用的GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


获取数据：

In [2]:
!wget https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt

--2026-02-15 09:42:52--  https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10797148 (10M) [text/plain]
Saving to: ‘train.txt’

train.txt           100%[===================>]  10.30M  --.-KB/s    in 0.02s   

2026-02-15 09:42:53 (468 MB/s) - ‘train.txt’ saved [10797148/10797148]



In [3]:
with open('train.txt', 'r') as f:
    text_data = f.read()
    
print(f"数据加载完成，字符长度: {len(text_data)}")

数据加载完成，字符长度: 10780437


## 第二步：工业级预处理 (Sub-sampling)

### 高频词下采样 (Sub-sampling):

我们要以此公式丢弃高频词：$P(w_i) = 1 - \sqrt{\frac{t}{f(w_i)}}$

其中 $t$ 是阈值（通常 $10^{-5}$），$f(w_i)$ 是词频。

这能让模型少学几次 "the"，多学几次 "learning"。

In [4]:
class Vocab:
    def __init__(self, text, min_freq=5):
        tokens = text.lower().split()
        self.counter = collections.Counter(tokens)
        
        # 1. 过滤低频词 (工业界通常过滤掉出现少于 5 次的词)
        self.tokens = [token for token in tokens if self.counter[token] >= min_freq]
        self.unique_tokens = sorted(list(set(self.tokens)))
        
        self.word_to_idx = {word: i for i, word in enumerate(self.unique_tokens)}
        self.idx_to_word = {i: word for i, word in enumerate(self.unique_tokens)}
        self.vocab_size = len(self.unique_tokens)
        
        # 计算总词数用于下采样
        total_count = len(self.tokens)
        self.freqs = {w: c/total_count for w, c in self.counter.items()}

    def subsample(self, t=1e-5):
        # 2. 下采样逻辑：频率越高，被丢弃的概率越大
        drop_prob = {w: 1 - math.sqrt(t / self.freqs[w]) for w in self.unique_tokens}
        
        # 生成最终的训练数据 (只保留没有被丢弃的词)
        self.train_tokens = [w for w in self.tokens if random.random() > drop_prob.get(w, 0)]
        print(f"下采样前词数: {len(self.tokens)}")
        print(f"下采样后词数: {len(self.train_tokens)}")

# 初始化词表
vocab = Vocab(text_data, min_freq=1) # 演示数据少，min_freq设为1
vocab.subsample()

下采样前词数: 2051910
下采样后词数: 502247


## 第三步：构建负采样数据集 (Negative Sampling Dataset)

这是最核心的变化。我们不再生成 (center, context),而是生成：(center, context, negatives)。

* Center: fox
* Context (正样本): jumps (标签为 1)
* Negatives (负样本): apple, car, sky... (标签为 0)

In [5]:
class Word2VecDataset(Dataset):
    def __init__(self, tokens, word_to_idx, vocab_size, window_size=3, num_negatives=5):
        self.tokens = [word_to_idx[w] for w in tokens]
        self.window_size = window_size
        self.num_negatives = num_negatives
        self.vocab_size = vocab_size
        
        # 预先生成负采样所需的权重 (基于词频的 0.75 次方)
        # 这是一个工业界的 Trick，让高频词稍微更容易被选为负样本
        word_counts = np.array([vocab.counter[vocab.idx_to_word[i]] for i in range(vocab_size)])
        word_freqs = word_counts / np.sum(word_counts)
        self.neg_sample_weights = torch.tensor(word_freqs ** 0.75)

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        # 获取中心词
        center_word = self.tokens[idx]
        
        # 获取正样本 (随机选窗口内的一个词)
        # 实际工业实现中，为了速度，通常随机选窗口内的一个，而不是遍历所有
        start = max(0, idx - self.window_size)
        end = min(len(self.tokens), idx + self.window_size + 1)
        context_words = self.tokens[start:idx] + self.tokens[idx+1:end]
        
        if len(context_words) == 0:
            return self.__getitem__((idx + 1) % len(self.tokens)) # 边界情况处理
            
        pos_word = random.choice(context_words)
        
        # 获取负样本 (Multinomial 采样)
        neg_words = torch.multinomial(self.neg_sample_weights, self.num_negatives, replacement=True)
        
        return torch.tensor(center_word), torch.tensor(pos_word), neg_words

# 建立 DataLoader
dataset = Word2VecDataset(vocab.train_tokens, vocab.word_to_idx, vocab.vocab_size)
dataloader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=4) # 调大 Batch Size

## 第四步：实现 Skip-gram 模型

之前的模型用的是 nn.Linear + Softmax。现在的模型用的是 双 Embedding 层 + LogSigmoid。

我们需要两个 Embedding 矩阵：

* in_embed:用来查中心词。
* out_embed: 用来查上下文词（和负样本词）。

这段代码利用了矩阵乘法 (torch.bmm) 并行计算所有样本的相似度。我们不再计算整个词表的 Softmax，只计算 1个正样本 + 5个负样本 的 Sigmoid。这就是让训练速度提升 1000 倍的秘诀。

In [6]:
class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(SkipGramNegSampling, self).__init__()
        # 中心词向量矩阵
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        # 上下文向量矩阵 (也叫输出矩阵)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)
        
        # 初始化权重 (这一步很重要，防止梯度消失/爆炸)
        self.in_embed.weight.data.uniform_(-0.5 / embed_dim, 0.5 / embed_dim)
        self.out_embed.weight.data.uniform_(-0.5 / embed_dim, 0.5 / embed_dim)

    def forward(self, center, context, negatives):
        # center: [batch_size]
        # context: [batch_size]
        # negatives: [batch_size, num_negatives]

        # 1. 获取向量
        # center_emb: [batch, 1, dim] -> 增加一个维度方便矩阵乘法
        center_emb = self.in_embed(center).unsqueeze(1)
        # context_emb: [batch, 1, dim]
        context_emb = self.out_embed(context).unsqueeze(1)
        # neg_emb: [batch, num_negatives, dim]
        neg_emb = self.out_embed(negatives)

        # 2. 计算正样本分数 (Maximize)
        # bmm = Batch Matrix Multiplication
        # [batch, 1, dim] * [batch, 1, dim]^T -> [batch, 1, 1]
        pos_score = torch.bmm(center_emb, context_emb.transpose(1, 2)).squeeze()
        pos_loss = -torch.nn.functional.logsigmoid(pos_score)

        # 3. 计算负样本分数 (Minimize)
        # 我们希望 center 和 negative 的向量点积越小越好
        # [batch, 1, dim] * [batch, neg, dim]^T -> [batch, 1, neg]
        neg_score = torch.bmm(center_emb, neg_emb.transpose(1, 2)).squeeze()
        # 注意这里的负号：我们希望 log_sigmoid(-score) 最大化
        neg_loss = -torch.nn.functional.logsigmoid(-neg_score)

        # 4. 总损失
        # sum() 是对 batch 内所有样本求和，mean() 是求平均
        return torch.mean(pos_loss + torch.sum(neg_loss, dim=1))

# 初始化模型
embed_dim = 100 # 工业级通常是 100-300
model = SkipGramNegSampling(vocab.vocab_size, embed_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.003)

## 第五步：高速训练循环

In [7]:
epochs = 10
print(f"开始训练，总批次: {len(dataloader)}")

for epoch in range(epochs):
    start_time = time.time()
    total_loss = 0
    
    for i, (center, context, negatives) in enumerate(dataloader):
        # 搬运数据到 GPU
        center = center.to(device)
        context = context.to(device)
        negatives = negatives.to(device)
        
        optimizer.zero_grad()
        
        # 计算 Loss
        loss = model(center, context, negatives)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if (i+1) % 100 == 0:
            print(f"Epoch {epoch+1}, Step {i+1}, Loss: {loss.item():.4f}")

    end_time = time.time()
    print(f"Epoch {epoch+1} 完成, 耗时: {end_time - start_time:.2f}s, 平均Loss: {total_loss/len(dataloader):.4f}")

开始训练，总批次: 981
Epoch 1, Step 100, Loss: 3.7475
Epoch 1, Step 200, Loss: 3.0523
Epoch 1, Step 300, Loss: 2.9003
Epoch 1, Step 400, Loss: 2.8542
Epoch 1, Step 500, Loss: 2.7822
Epoch 1, Step 600, Loss: 2.6946
Epoch 1, Step 700, Loss: 2.6746
Epoch 1, Step 800, Loss: 2.6789
Epoch 1, Step 900, Loss: 2.6534
Epoch 1 完成, 耗时: 13.54s, 平均Loss: 2.9468
Epoch 2, Step 100, Loss: 2.6399
Epoch 2, Step 200, Loss: 2.6108
Epoch 2, Step 300, Loss: 2.6322
Epoch 2, Step 400, Loss: 2.6545
Epoch 2, Step 500, Loss: 2.6716
Epoch 2, Step 600, Loss: 2.6270
Epoch 2, Step 700, Loss: 2.6581
Epoch 2, Step 800, Loss: 2.6203
Epoch 2, Step 900, Loss: 2.6554
Epoch 2 完成, 耗时: 12.82s, 平均Loss: 2.6507
Epoch 3, Step 100, Loss: 2.6233
Epoch 3, Step 200, Loss: 2.6086
Epoch 3, Step 300, Loss: 2.5721
Epoch 3, Step 400, Loss: 2.5495
Epoch 3, Step 500, Loss: 2.6057
Epoch 3, Step 600, Loss: 2.5745
Epoch 3, Step 700, Loss: 2.5207
Epoch 3, Step 800, Loss: 2.5501
Epoch 3, Step 900, Loss: 2.5451
Epoch 3 完成, 耗时: 12.49s, 平均Loss: 2.5616
Epoch

## 第六步：工业级评估 (类比测试)

在工业界，我们很少看 Loss，我们看 Analogy Task (类比任务)。

比如：King - Man + Woman = `?`

In [8]:
def get_embedding(word):
    # 提取 Embedding，注意：有时候会把 in_embed 和 out_embed 加上或者平均
    # 但通常只取 in_embed 也可以
    word_idx = vocab.word_to_idx[word]
    return model.in_embed.weight[word_idx].cpu().detach().numpy()

def find_analogy(w1, w2, w3):
    # w1 - w2 + w3 = ? 
    # e.g., king - man + woman = queen
    v1 = get_embedding(w1)
    v2 = get_embedding(w2)
    v3 = get_embedding(w3)
    
    target_vec = v1 - v2 + v3
    
    # 计算与所有词的相似度
    all_vecs = model.in_embed.weight.cpu().detach().numpy()
    similarities = np.dot(all_vecs, target_vec) / (np.linalg.norm(all_vecs, axis=1) * np.linalg.norm(target_vec))
    
    # 排序
    top_indices = np.argsort(similarities)[::-1]
    
    # 打印前5个结果 (排除输入的词本身)
    input_words = set([w1, w2, w3])
    found_words = []
    for idx in top_indices:
        word = vocab.idx_to_word[idx]
        if word not in input_words:
            found_words.append(word)
        if len(found_words) >= 3:
            break
            
    return found_words

try:
    print("King - Man + Woman =", find_analogy('king', 'man', 'woman'))
except:
    print("词表中可能没有这些词，请使用真实的大规模语料库。")

# 简单相似度
try:
    print("Intelligence 相似词:", find_analogy('intelligence', 'artificial', 'intelligence')) # 这里的逻辑只是为了调用查找函数
except:
    pass

King - Man + Woman = ['vizier', 'vasa', 'filiation']
Intelligence 相似词: ['fits', 'macv', 'scientologists']
